In [2]:
# pip install pymupdf4llm pymupdf rapidocr-onnxruntime pytesseract

import pymupdf4llm
import os

# Create a directory to store extracted images
image_folder = "pdf_extractor\\extracted_images"
os.makedirs(image_folder, exist_ok=True)

# Extract markdown and save images
md_text = pymupdf4llm.to_markdown(
    doc="test_pdfs\\01030000000008.pdf",
    write_images=True,          # Tells the library to extract images
    image_path=image_folder     # Where to save the image files
)

print(md_text)


=== Document parser messages ===
Using Tesseract for OCR processing.

73 

Circulating Things, Circulating Stereotypes 

indicates the use of balsam, which is “indigenous in various parts of Arabia,” as an ingredient in the “Myrabolan comfit.”25 Such references emphasize Arabia’s exoticism and refined taste, as well as the sweetness and fragrance of its products, which were much valued during a time when the consumption of sugar and spices was rising rapidly among European populations. 

Coffee is another staple thing customarily associated with the area. In his _Dictionary,_ Johnson indicates the Arabic origin of coffee and rightly so, as one the most popular types of coffee is called “Arabica” because it was first domesticated for commercial use in the southern part of Arabia the Happy (present-day Yemen). Given the Muslim prohibition of alcohol, coffee became particularly attractive to the Muslim world as “the wine of Islam,”26 and spread through the ports of the Persian Gulf in Wes

In [4]:
import fitz  # PyMuPDF (installed automatically with pymupdf4llm)

def extract_cv_layout(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []
    
    for page in doc:
        # get_text("words") returns a list of tuples for every word:
        # (x0, y0, x1, y1, "word", block_no, line_no, word_no)
        words = page.get_text("words")
        
        if not words:
            continue
            
        # 1. Sort words top-to-bottom, then left-to-right
        # We round the y0 coordinate by dividing by 10. 
        # This creates a "tolerance" of ~10 pixels, grouping words that 
        # are on the same line but might be misaligned by a fraction of a millimeter.
        words.sort(key=lambda w: (round(w[1] / 10), w[0]))
        
        current_y = round(words[0][1] / 10)
        current_line = []
        prev_x1 = None
        
        for w in words:
            word_text = w[4]
            word_y = round(w[1] / 10)
            word_x0 = w[0]
            
            # If we moved down the page, save the current line and start a new one
            if word_y != current_y:
                full_text.append(" ".join(current_line))
                current_line = []
                current_y = word_y
                prev_x1 = None
                
            # 2. Detect large horizontal gaps on the same line
            # If the gap between the last word and this word is larger than 20 pixels,
            # insert a semantic separator (like a pipe or dash) so the LLM knows they were separated.
            if prev_x1 is not None and (word_x0 - prev_x1) > 5:
                current_line.append("|")
                
            current_line.append(word_text)
            prev_x1 = w[2] # Update previous end coordinate to current word's end
            
        # Append the very last line
        if current_line:
            full_text.append(" ".join(current_line))
            
    return "\n".join(full_text)

# Run the extraction
cv_text = extract_cv_layout("test_pdfs\\01030000000008.pdf")
print(cv_text)

Circulating Things, Circulating Stereotypes | 73
­indicates the use of balsam, which is “indigenous
in various parts of Arabia,” as an ingredient in the
“Myrabolan comfit.”25 Such references emphasize
Arabia’s exoticism and refined taste, as well as the
sweetness and fragrance of | its products, which
were much valued during a time when the con-
sumption of sugar and spices was rising rapidly
among European populations.
Coffee is another staple thing customarily asso-
ciated with the area. In his Dictionary, Johnson indi-
cates the Arabic origin of coffee and rightly so, as
one the most popular types of coffee is called “Ara-
bica” because it was first domesticated for commer-
cial use in the southern part of Arabia the Happy | Figure 4.2 | William Hogarth, Taste in High Life [graphic].
Print made by isaac mills after William
(present-day Yemen). Given the Muslim prohibi-
tion of alcohol, coffee became particularly attrac- | Hogarth’s painting, without the artist’s
tive to the Muslim w

In [5]:
import fitz  # PyMuPDF

def get_image_metadata(image_bytes):
    """
    Placeholder for your Vision Model API call.
    In production, you would send 'image_bytes' to Gemini, OpenAI, etc.
    """
    # Example: response = client.generate_content(["Describe this image", image_bytes])
    return "[Image Insight: A bar chart showing Q3 revenue growth of 15%]"


def multimodal_pdf_parser(pdf_path):
    doc = fitz.open(pdf_path)
    final_document_text = []

    for page_num, page in enumerate(doc):
        page_elements = []

        # 1. Extract Images and their coordinates
        # get_image_info(xrefs=True) gives us the bounding box on the page
        image_info_list = page.get_image_info(xrefs=True)
        for img in image_info_list:
            xref = img["xref"]
            bbox = img["bbox"]  # (x0, y0, x1, y1)
            
            # Extract the actual image bytes to send to the AI
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            page_elements.append({
                "type": "image",
                "y0": bbox[1],       # Top-Y coordinate for sorting
                "content": image_bytes
            })

        # 2. Extract Text Blocks and their coordinates
        blocks = page.get_text("blocks")
        for b in blocks:
            # block format: (x0, y0, x1, y1, "text", block_no, block_type)
            # block_type 0 is text, block_type 1 is image
            if b[6] == 0:
                page_elements.append({
                    "type": "text",
                    "y0": b[1],      # Top-Y coordinate for sorting
                    "content": b[4].strip()
                })

        # 3. Sort all elements top-to-bottom using the y0 coordinate
        page_elements.sort(key=lambda e: e["y0"])

        # 4. Assemble the reading order
        final_document_text.append(f"--- START OF PAGE {page_num + 1} ---")
        
        for element in page_elements:
            if element["type"] == "text":
                # It's standard text, add it to our output
                final_document_text.append(element["content"])
                
            elif element["type"] == "image":
                # It's an image, query the vision model and insert the description
                vision_description = get_image_metadata(element["content"])
                final_document_text.append(f"\n{vision_description}\n")

    return "\n\n".join(final_document_text)

# Run the pipeline
parsed_content = multimodal_pdf_parser("test_pdfs\\01030000000008.pdf")
print(parsed_content)

--- START OF PAGE 1 ---

Circulating Things, Circulating Stereotypes
73


[Image Insight: A bar chart showing Q3 revenue growth of 15%]


­indicates the use of balsam, which is “indigenous 
in various parts of Arabia,” as an ingredient in the 
“Myrabolan comfit.”25 Such references emphasize 
Arabia’s exoticism and refined taste, as well as the 
sweetness and fragrance of its products, which 
were much valued during a time when the con-
sumption of sugar and spices was rising rapidly 
among European populations.

Coffee is another staple thing customarily asso-
ciated with the area. In his Dictionary, Johnson indi-
cates the Arabic origin of coffee and rightly so, as 
one the most popular types of coffee is called “Ara-
bica” because it was first domesticated for commer-
cial use in the southern part of Arabia the Happy 
(present-day Yemen). Given the Muslim prohibi-
tion of alcohol, coffee became particularly attrac-
tive to the Muslim world as “the wine of Islam,”26 
and spread throug

In [7]:
import fitz  # PyMuPDF


def get_image_metadata(image_bytes: bytes) -> str:
    """
    Placeholder: Send image bytes to your Vision Model (Gemini, GPT-4o, etc.)
    and return the generated caption/metadata.
    """
    # Example Vision Call:
    # response = client.models.generate_content(
    #     model='gemini-2.5-flash',
    #     contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/png"), "Describe this image concise and accurately."]
    # )
    # return response.text
    return "[Image Metadata: Logo / Chart / Profile Photo extracted]"


def parse_multimodal_layout_pdf(
    pdf_path: str,
    y_tolerance: int = 10,       # Max vertical pixel variance for words on the same line
    x_gap_threshold: int = 5,   # Horizontal pixel gap needed to insert a '|' separator
    min_image_dim: int = 30      # Ignore images smaller than this (e.g. icons, bullet points)
) -> str:
    
    doc = fitz.open(pdf_path)
    final_output = []

    for page_num, page in enumerate(doc):
        page_elements = []

        # =====================================================================
        # STEP 1: Extract Images + Coordinates
        # =====================================================================
        image_info_list = page.get_image_info(xrefs=True)
        for img in image_info_list:
            xref = img["xref"]
            bbox = img["bbox"]  # (x0, y0, x1, y1)
            
            # Optional: Filter out tiny decorative icons/logos to save vision API costs
            width = bbox[2] - bbox[0]
            height = bbox[3] - bbox[1]
            if width < min_image_dim or height < min_image_dim:
                continue
            
            # Extract raw image bytes
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            page_elements.append({
                "type": "image",
                "y0": bbox[1],  # Use top vertical coordinate for page ordering
                "content": image_bytes
            })

        # =====================================================================
        # STEP 2: Extract Words, Group into Lines, & Detect Horizontal Gaps
        # =====================================================================
        words = page.get_text("words")
        # Format of word tuple: (x0, y0, x1, y1, "word_text", block_no, line_no, word_no)
        
        if words:
            # 2a. Group words into vertical line "buckets" using Y-tolerance
            lines_by_y = {}
            for w in words:
                y_bucket = round(w[1] / y_tolerance)
                lines_by_y.setdefault(y_bucket, []).append(w)

            # 2b. Build text lines with horizontal gap '|' markers
            for y_bucket, line_words in lines_by_y.items():
                # Sort words within the line left-to-right
                line_words.sort(key=lambda w: w[0])
                
                line_parts = []
                prev_x1 = None
                # Representative Y position for sorting the entire line on the page
                line_y0 = line_words[0][1] 

                for w in line_words:
                    word_str = w[4]
                    word_x0 = w[0]
                    
                    # Insert a pipe '|' if the gap between adjacent words is wide
                    if prev_x1 is not None and (word_x0 - prev_x1) > x_gap_threshold:
                        line_parts.append("|")
                    
                    line_parts.append(word_str)
                    prev_x1 = w[2]

                line_string = " ".join(line_parts)
                
                page_elements.append({
                    "type": "text",
                    "y0": line_y0,
                    "content": line_string
                })

        # =====================================================================
        # STEP 3: Combine & Sort Everything Top-to-Bottom
        # =====================================================================
        page_elements.sort(key=lambda element: element["y0"])

        # =====================================================================
        # STEP 4: Render Page Lines & Call Vision Model for Images
        # =====================================================================
        final_output.append(f"--- START OF PAGE {page_num + 1} ---")
        
        for element in page_elements:
            if element["type"] == "text":
                final_output.append(element["content"])
                
            elif element["type"] == "image":
                # Process image through Vision API
                vision_metadata = get_image_metadata(element["content"])
                final_output.append(f"\n{vision_metadata}\n")

    return "\n".join(final_output)


# --- Execution Example ---
if __name__ == "__main__":
    extracted_doc = parse_multimodal_layout_pdf("test_pdfs\\01030000000008.pdf")
    print(extracted_doc)

--- START OF PAGE 1 ---
Circulating Things, Circulating Stereotypes | 73

[Image Metadata: Logo / Chart / Profile Photo extracted]

­indicates the use of balsam, which is “indigenous
in various parts of Arabia,” as an ingredient in the
“Myrabolan comfit.”25 Such references emphasize
Arabia’s exoticism and refined taste, as well as the
sweetness and fragrance of | its products, which
were much valued during a time when the con-
sumption of sugar and spices was rising rapidly
among European populations.
Coffee is another staple thing customarily asso-
ciated with the area. In his Dictionary, Johnson indi-
cates the Arabic origin of coffee and rightly so, as
one the most popular types of coffee is called “Ara-
bica” because it was first domesticated for commer-
cial use in the southern part of Arabia the Happy | Figure 4.2 | William Hogarth, Taste in High Life [graphic].
Print made by isaac mills after William
(present-day Yemen). Given the Muslim prohibi-
tion of alcohol, coffee became p

In [8]:
# --- Execution Example ---
if __name__ == "__main__":
    extracted_doc = parse_multimodal_layout_pdf("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf")
    print(extracted_doc)

--- START OF PAGE 1 ---
AMD | Developer | Hackathon: | Participant
Submission | Guide
This document covers everything you need to build and submit a competitive entry. Exact
evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
hardcoded to specific answers.
Track 1: General-Purpose AI Agent
What you are building
An AI agent that handles a wide variety of natural language tasks across multiple capability
domains, using Fireworks AI models as efficiently as possible.
Why this task exists:
Enterprises want to control AI spend without sacrificing user experience: not every task needs a
premium proprietary model. A common pattern is hosting a range of models in-house
(open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
necessary. Track 1 asks you to build that smart router: run as many local models as you need,
they cost zero toward your score, and make as few external Fireworks API calls as possible
while still clearing

In [9]:
import fitz  # PyMuPDF

def extract_with_tables(pdf_path, y_tolerance=10, x_gap_threshold=25):
    doc = fitz.open(pdf_path)
    final_output = []

    for page_num, page in enumerate(doc):
        page_elements = []
        
        # =====================================================================
        # STEP 1: Detect and Extract Tables First
        # =====================================================================
        tables = page.find_tables()
        table_bboxes = []
        
        for tab in tables:
            # Store the bounding box so we can ignore free text inside it later
            table_bboxes.append(tab.bbox)
            
            # Extract data. tab.extract() returns a list of lists.
            # It automatically handles multi-line cells like in image_fac3ba.png
            extracted_data = tab.extract()
            md_rows = []
            
            for i, row in enumerate(extracted_data):
                # Clean newlines inside multi-line cells to keep Markdown intact
                clean_row = [str(cell).replace('\n', ' ').strip() if cell else "" for cell in row]
                md_rows.append("| " + " | ".join(clean_row) + " |")
                
                # Add markdown header separator after the first row
                if i == 0:
                    md_rows.append("|" + "|".join(["---"] * len(row)) + "|")
                    
            table_string = "\n".join(md_rows)
            
            page_elements.append({
                "type": "table",
                "y0": tab.bbox[1], # Top Y coordinate for sorting
                "content": table_string
            })

        # =====================================================================
        # STEP 2: Extract Words (Skipping Table Regions)
        # =====================================================================
        words = page.get_text("words")
        
        if words:
            lines_by_y = {}
            for w in words:
                # Create a rectangle for the current word
                word_rect = fitz.Rect(w[:4])
                
                # Check if this word is inside any of our detected tables
                in_table = any(word_rect.intersects(t_box) for t_box in table_bboxes)
                
                # Only process free text that is NOT in a table
                if not in_table:
                    y_bucket = round(w[1] / y_tolerance)
                    lines_by_y.setdefault(y_bucket, []).append(w)

            # Build text lines with horizontal gap '|' markers for the remaining text
            for y_bucket, line_words in lines_by_y.items():
                line_words.sort(key=lambda w: w[0])
                line_parts = []
                prev_x1 = None
                line_y0 = line_words[0][1] 

                for w in line_words:
                    word_str = w[4]
                    word_x0 = w[0]
                    
                    if prev_x1 is not None and (word_x0 - prev_x1) > x_gap_threshold:
                        line_parts.append("|")
                    
                    line_parts.append(word_str)
                    prev_x1 = w[2]

                page_elements.append({
                    "type": "text",
                    "y0": line_y0,
                    "content": " ".join(line_parts)
                })

        # =====================================================================
        # STEP 3: Sort Everything Top-to-Bottom and Render
        # =====================================================================
        page_elements.sort(key=lambda element: element["y0"])

        final_output.append(f"--- START OF PAGE {page_num + 1} ---")
        
        for element in page_elements:
            # Add spacing around tables for cleaner output
            if element["type"] == "table":
                final_output.append(f"\n{element['content']}\n")
            else:
                final_output.append(element["content"])

    return "\n".join(final_output)

# Run the updated parser
print(extract_with_tables("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf"))

--- START OF PAGE 1 ---
AMD Developer Hackathon: Participant
Submission Guide
This document covers everything you need to build and submit a competitive entry. Exact
evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
hardcoded to specific answers.
Track 1: General-Purpose AI Agent
What you are building
An AI agent that handles a wide variety of natural language tasks across multiple capability
domains, using Fireworks AI models as efficiently as possible.
Why this task exists:
Enterprises want to control AI spend without sacrificing user experience: not every task needs a
premium proprietary model. A common pattern is hosting a range of models in-house
(open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
necessary. Track 1 asks you to build that smart router: run as many local models as you need,
they cost zero toward your score, and make as few external Fireworks API calls as possible
while still clearing the acc

In [24]:
import fitz  # PyMuPDF


def get_image_metadata(image_bytes: bytes) -> str:
    """
    Placeholder: Send image bytes to your Vision Model (Gemini, GPT-4o, etc.)
    and return the generated caption/metadata.
    """
    # Example:
    # response = client.models.generate_content(
    #     model='gemini-2.5-flash',
    #     contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/png"), "Describe this image accurately."]
    # )
    # return response.text
    return "[Image Metadata: Logo / Profile Photo / Diagram extracted]"


def parse_advanced_pdf(
    pdf_path: str,
    y_tolerance: int = 10,       # Vertical tolerance for grouping text onto the same line
    x_gap_threshold: int = 25,   # Horizontal gap distance to insert a '|' column separator
    min_image_dim: int = 30      # Ignore icons or tiny graphics smaller than this size
) -> str:
    
    doc = fitz.open(pdf_path)
    final_output = []

    for page_num, page in enumerate(doc):
        page_elements = []

        # =====================================================================
        # STEP 1: Detect and Extract Tables First
        # =====================================================================
        tables = page.find_tables()
        table_bboxes = []
        
        for tab in tables:
            # 🛑 FIX: Skip empty phantom tables to prevent ValueError
            if not tab.cells:
                continue
                
            table_bboxes.append(tab.bbox)
            extracted_data = tab.extract()
            md_rows = []
            
            for i, row in enumerate(extracted_data):
                # Clean up newlines inside multi-line cells for pristine markdown table layout
                clean_row = [str(cell).replace('\n', ' ').strip() if cell else "" for cell in row]
                md_rows.append("| " + " | ".join(clean_row) + " |")
                
                # Add markdown table header separator after the primary header row
                if i == 0:
                    md_rows.append("|" + "|".join(["---"] * len(row)) + "|")
                    
            table_string = "\n".join(md_rows)
            
            page_elements.append({
                "type": "table",
                "y0": tab.bbox[1],
                "content": f"\n{table_string}\n"
            })

        # =====================================================================
        # STEP 2: Extract Images & Coordinates (Skipping tiny icons)
        # =====================================================================
        image_info_list = page.get_image_info(xrefs=True)
        for img in image_info_list:
            bbox = img["bbox"]
            width = bbox[2] - bbox[0]
            height = bbox[3] - bbox[1]
            
            if width < min_image_dim or height < min_image_dim:
                continue
            
            base_image = doc.extract_image(img["xref"])
            image_bytes = base_image["image"]
            
            page_elements.append({
                "type": "image",
                "y0": bbox[1],
                "content": image_bytes
            })

        # =====================================================================
        # STEP 3: Extract Formatted Text via Dict (Skipping Table Areas)
        # =====================================================================
        text_dict = page.get_text("dict")
        lines_by_y = {}
        
        for block in text_dict.get("blocks", []):
            if block.get("type") != 0:  # Type 0 is text blocks
                continue
                
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text").strip()
                    if not text:
                        continue
                    
                    bbox = span.get("bbox")  # (x0, y0, x1, y1)
                    span_rect = fitz.Rect(bbox)
                    
                    # Ignore text elements that reside inside detected tables
                    if any(span_rect.intersects(t_box) for t_box in table_bboxes):
                        continue
                    
                    # Detect Font Formatting properties
                    font_size = span.get("size")
                    font_name = span.get("font", "").lower()
                    flags = span.get("flags")
                    
                    is_bold = (flags & 2**4) or ("bold" in font_name)
                    is_header = font_size > 14
                    
                    # Wrap text in Markdown syntax based on font attributes
                    if is_bold and not is_header:
                        text = f"**{text}**"
                    elif is_header:
                        if font_size > 20:
                            text = f"# {text}"
                        elif font_size > 16:
                            text = f"## {text}"
                        else:
                            text = f"### {text}"

                    # Group by vertical Y-bucket coordinate
                    y_bucket = round(bbox[1] / y_tolerance)
                    lines_by_y.setdefault(y_bucket, []).append((bbox[0], bbox[2], text))

        # Reconstruct text lines with horizontal gap '|' separators
        for y_bucket, spans in lines_by_y.items():
            spans.sort(key=lambda s: s[0])  # Sort left-to-right
            
            line_parts = []
            prev_x1 = None
            line_y0 = y_bucket * y_tolerance

            for x0, x1, formatted_text in spans:
                # Insert pipe if there is a wide whitespace separation (e.g., CV columns)
                if prev_x1 is not None and (x0 - prev_x1) > x_gap_threshold:
                    line_parts.append("|")
                
                line_parts.append(formatted_text)
                prev_x1 = x1
                
            page_elements.append({
                "type": "text",
                "y0": line_y0,
                "content": " ".join(line_parts)
            })

        # =====================================================================
        # STEP 4: Sort All Elements Top-to-Bottom and Build Page Output
        # =====================================================================
        page_elements.sort(key=lambda element: element["y0"])

        final_output.append(f"--- START OF PAGE {page_num + 1} ---")
        
        for element in page_elements:
            if element["type"] in ["text", "table"]:
                final_output.append(element["content"])
            elif element["type"] == "image":
                vision_description = get_image_metadata(element["content"])
                final_output.append(f"\n{vision_description}\n")

    return "\n".join(final_output)


# --- Execution Example ---
if __name__ == "__main__":
    markdown_result = parse_advanced_pdf("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf")
    print(markdown_result)

--- START OF PAGE 1 ---
## AMD Developer Hackathon: Participant
## Submission Guide
This document covers everything you need to build and submit a competitive entry. Exact
evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
hardcoded to specific answers.
### Track 1: General-Purpose AI Agent
What you are building
An AI agent that handles a wide variety of natural language tasks across multiple capability
domains, using Fireworks AI models as efficiently as possible.
Why this task exists:
Enterprises want to control AI spend without sacrificing user experience: not every task needs a
premium proprietary model. A common pattern is hosting a range of models in-house
(open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
necessary. Track 1 asks you to build that smart router: run as many local models as you need,
they cost zero toward your score, and make as few external Fireworks API calls as possible
while still cleari

In [25]:
# --- Execution Example ---
if __name__ == "__main__":
    markdown_result = parse_advanced_pdf("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")
    print(markdown_result)

--- START OF PAGE 1 ---
# GBP Statement
Generated on Jul 30, 2026
Revolut Bank UK Ltd
PRASAD CHATHURA MANAMALA MUDIYANSELAGE
77 Marlborough Grove | Sort Code | 230120
Flat 33, Press Court | Account Number 66593646
SE1 5JU
London | IBAN | GB31REVO23012066593646
BIC | REVOGB21
Balance summary
Closing
Product | Opening balance Money out | Money in
balance
Account (Current Account) | £9.56 | £236.62 | £248.21 | £21.15
Total | £9.56 | £236.62 | £248.21 | £21.15
The balance on your statement might differ from the balance shown in your app. The statement balance only reflects completed transactions, while the app shows the balance available for use, which accounts for pending transactions.
Account transactions from June 1, 2026 to July 30, 2026
Date | Description | Money out | Money in | Balance
Jun 13, 2026 | Transport for London | £3.50 | £6.06
To: Tfl Travel Ch, Tfl.gov.uk/cp
Card: 535456******9135
Jun 13, 2026 | Payment from L JAYAWEERA-ARACHC | £10.00 | £16.06
Reference: HUBBY
From: L JA

In [32]:
import fitz  # PyMuPDF


def get_image_metadata(image_bytes: bytes) -> str:
    """
    Placeholder: Send image bytes to your Vision Model (Gemini, GPT-4o, etc.)
    and return the generated caption/metadata.
    """
    return "[Image Metadata: Extracted visual content]"


def parse_advanced_pdf(
    pdf_path: str,
    y_tolerance: int = 10,       
    x_gap_threshold: int = 5,   
    min_image_dim: int = 30      
) -> str:
    
    doc = fitz.open(pdf_path)
    final_output = []

    for page_num, page in enumerate(doc):
        page_elements = []

        # =====================================================================
        # STEP 1: General Borderless Table Detection (Dynamic Clustering)
        # =====================================================================
        table_bboxes = []  # 🛑 FIX: Initialize the list to store table coordinates
        
        initial_tables = page.find_tables(strategy="text")
        
        for tab in initial_tables:
            if not tab.cells:
                continue
                
            # 🛑 FIX: Store the bounding box so STEP 3 knows to skip text in this area
            table_bboxes.append(tab.bbox) 
            
            words = page.get_text("words", clip=tab.bbox)
            if not words:
                continue

            x0_coords = sorted([w[0] for w in words])
            
            dynamic_vertical_lines = []
            current_x = x0_coords[0]
            dynamic_vertical_lines.append(current_x - 2)
            
            for x in x0_coords:
                if x - current_x > 20: 
                    current_x = x
                    dynamic_vertical_lines.append(current_x - 2)
                    
            table_settings = {
                "vertical_strategy": "explicit",
                "vertical_lines": dynamic_vertical_lines,
                "horizontal_strategy": "text"
            }
            
            refined_tables = page.find_tables(clip=tab.bbox, **table_settings)
            
            if not refined_tables or not refined_tables[0].cells:
                continue
                
            refined_tab = refined_tables[0]
            extracted_data = refined_tab.extract() 
            md_rows = []
            
            for i, row in enumerate(extracted_data):
                clean_row = [str(cell).replace('\n', ' ').strip() if cell else "" for cell in row]
                md_rows.append("| " + " | ".join(clean_row) + " |")
                
                if i == 0:
                    md_rows.append("|" + "|".join(["---"] * len(row)) + "|")
                    
            table_string = "\n".join(md_rows)
            
            page_elements.append({
                "type": "table",
                "y0": tab.bbox[1],
                "content": f"\n{table_string}\n"
            })

        # =====================================================================
        # STEP 2: Extract Images & Coordinates (Skipping tiny icons)
        # =====================================================================
        image_info_list = page.get_image_info(xrefs=True)
        for img in image_info_list:
            bbox = img["bbox"]
            width = bbox[2] - bbox[0]
            height = bbox[3] - bbox[1]
            
            if width < min_image_dim or height < min_image_dim:
                continue
            
            base_image = doc.extract_image(img["xref"])
            image_bytes = base_image["image"]
            
            page_elements.append({
                "type": "image",
                "y0": bbox[1],
                "content": image_bytes
            })

        # =====================================================================
        # STEP 3: Extract Formatted Text via Dict (Skipping Table Areas)
        # =====================================================================
        text_dict = page.get_text("dict")
        lines_by_y = {}
        
        for block in text_dict.get("blocks", []):
            if block.get("type") != 0: 
                continue
                
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text").strip()
                    if not text:
                        continue
                    
                    bbox = span.get("bbox")  
                    span_rect = fitz.Rect(bbox)
                    
                    # Ignore text elements that reside inside detected tables
                    if any(span_rect.intersects(t_box) for t_box in table_bboxes):
                        continue
                    
                    font_size = span.get("size")
                    font_name = span.get("font", "").lower()
                    flags = span.get("flags")
                    
                    is_bold = (flags & 2**4) or ("bold" in font_name)
                    is_header = font_size > 14
                    
                    if is_bold and not is_header:
                        text = f"**{text}**"
                    elif is_header:
                        if font_size > 20:
                            text = f"# {text}"
                        elif font_size > 16:
                            text = f"## {text}"
                        else:
                            text = f"### {text}"

                    y_bucket = round(bbox[1] / y_tolerance)
                    lines_by_y.setdefault(y_bucket, []).append((bbox[0], bbox[2], text))

        for y_bucket, spans in lines_by_y.items():
            spans.sort(key=lambda s: s[0])  
            
            line_parts = []
            prev_x1 = None
            line_y0 = y_bucket * y_tolerance

            for x0, x1, formatted_text in spans:
                if prev_x1 is not None and (x0 - prev_x1) > x_gap_threshold:
                    line_parts.append("|")
                
                line_parts.append(formatted_text)
                prev_x1 = x1
                
            page_elements.append({
                "type": "text",
                "y0": line_y0,
                "content": " ".join(line_parts)
            })

        # =====================================================================
        # STEP 4: Sort All Elements Top-to-Bottom and Build Page Output
        # =====================================================================
        page_elements.sort(key=lambda element: element["y0"])

        final_output.append(f"--- START OF PAGE {page_num + 1} ---")
        
        for element in page_elements:
            if element["type"] in ["text", "table"]:
                final_output.append(element["content"])
            elif element["type"] == "image":
                vision_description = get_image_metadata(element["content"])
                final_output.append(f"\n{vision_description}\n")

    return "\n".join(final_output)


# --- Execution Example ---
if __name__ == "__main__":
    markdown_result = parse_advanced_pdf("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf")
    print(markdown_result)

--- START OF PAGE 1 ---
## AMD Developer Hackathon: Participant
## Submission Guide
This document covers everything you need to build and submit a competitive entry. Exact
evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
hardcoded to specific answers.
### Track 1: General-Purpose AI Agent
What you are building
An AI agent that handles a wide variety of natural language tasks across multiple capability
domains, using Fireworks AI models as efficiently as possible.
Why this task exists:
Enterprises want to control AI spend without sacrificing user experience: not every task needs a
premium proprietary model. A common pattern is hosting a range of models in-house
(open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
necessary. Track 1 asks you to build that smart router: run as many local models as you need,
they cost zero toward your score, and make as few external Fireworks API calls as possible
while still cleari

In [33]:
# --- Execution Example ---
if __name__ == "__main__":
    markdown_result = parse_advanced_pdf("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")
    print(markdown_result)

--- START OF PAGE 1 ---
# GBP Statement
Generated on Jul 30, 2026
Revolut Bank UK Ltd
PRASAD CHATHURA MANAMALA MUDIYANSELAGE
77 Marlborough Grove | Sort Code | 230120
Flat 33, Press Court | Account Number 66593646
SE1 5JU
London | IBAN | GB31REVO23012066593646
BIC | REVOGB21
Balance summary
Closing
Product | Opening balance | Money out | Money in
balance
Account (Current Account) | £9.56 | £236.62 | £248.21 | £21.15
Total | £9.56 | £236.62 | £248.21 | £21.15
The balance on your statement might differ from the balance shown in your app. The statement balance only reflects completed transactions, while the app shows the balance available for use, which accounts for pending transactions.
Account transactions from June 1, 2026 to July 30, 2026
Date | Description | Money out | Money in | Balance
Jun 13, 2026 | Transport for London | £3.50 | £6.06
To: Tfl Travel Ch, Tfl.gov.uk/cp
Card: 535456******9135
Jun 13, 2026 | Payment from L JAYAWEERA-ARACHC | £10.00 | £16.06
Reference: HUBBY
From: L 

In [42]:
import pandas as pd
import pymupdf


def universal_borderless_extractor(pdf_path, page_num=0):
    doc = pymupdf.open(pdf_path)
    page = doc[page_num]

    # 1. Grab every single word token with detailed structural blocks
    words_data = page.get_text("words")
    if not words_data:
        doc.close()
        return pd.DataFrame()

    # 2. Build a structural row mapper using strict baseline tracking
    rows_map = {}
    y_threshold = 3  # Maximum vertical delta for row grouping

    for w in words_data:
        x0, y0, x1, y1, word_text = w[:5]
        y_mid = (y0 + y1) / 2

        matched_row_y = None
        for established_y in rows_map:
            if abs(y_mid - established_y) <= y_threshold:
                matched_row_y = established_y
                break

        word_node = {"x0": x0, "x1": x1, "text": word_text}
        if matched_row_y is not None:
            rows_map[matched_row_y].append(word_node)
        else:
            rows_map[y_mid] = [word_node]

    # Order rows sequentially from top to bottom
    sorted_y_levels = sorted(rows_map.keys())

    # 3. STATISTICAL SWEEP: Identify where text NEVER crosses horizontally
    page_width = int(page.rect.x1)
    horizontal_occupancy = [0] * (page_width + 1)

    # Fill an array tracing where text blocks physically sit across the X-axis
    for y in sorted_y_levels:
        for word in rows_map[y]:
            start_x = max(0, int(word["x0"]))
            end_x = min(page_width, int(word["x1"]))
            for x in range(start_x, end_x + 1):
                horizontal_occupancy[x] += 1

    # Locate empty gaps (white space alleys) running vertically down the page
    column_boundaries = []
    in_gap = True
    gap_start = 0

    for x in range(0, page_width + 1):
        # We define a structural column boundary if a space remains consistently clear
        has_text_presence = horizontal_occupancy[x] > 0

        if in_gap and has_text_presence:
            # Gap ends -> text column begins
            gap_end = x
            if len(column_boundaries) > 0:
                # Set a divider line right in the middle of the empty space
                column_boundaries[-1]["right"] = (gap_start + gap_end) / 2
            column_boundaries.append({"left": (gap_start + gap_end) / 2, "right": page_width})
            in_gap = False
        elif not in_gap and not has_text_presence:
            # Text column ends -> white space gap begins
            gap_start = x
            in_gap = True

    # 4. CONSTRUCT DATA ROWS: Route characters safely into their vertical boxes
    structured_matrix = []

    for y in sorted_y_levels:
        line_items = rows_map[y]
        line_items.sort(key=lambda item: item["x0"])

        # Merge split token segments inside the row
        merged_line_tokens = []
        for token in line_items:
            if not merged_line_tokens:
                merged_line_tokens.append(token)
            else:
                last_node = merged_line_tokens[-1]
                # Combine tokens separated by small sub-word tracking distances
                if token["x0"] - last_node["x1"] < 8:
                    last_node["text"] += f" {token['text']}"
                    last_node["x1"] = token["x1"]
                else:
                    merged_line_tokens.append(token)

        # Map merged row entries directly to our column buckets
        row_cells = [""] * len(column_boundaries)
        has_row_data = False

        for token in merged_line_tokens:
            token_center = (token["x0"] + token["x1"]) / 2

            for col_idx, boundary in enumerate(column_boundaries):
                if boundary["left"] <= token_center <= boundary["right"]:
                    row_cells[col_idx] = token["text"]
                    has_row_data = True
                    break

        if has_row_data:
            structured_matrix.append(row_cells)

    doc.close()

    # 5. GENERATE DATA FRAME
    if not structured_matrix:
        return pd.DataFrame()

    df = pd.DataFrame(structured_matrix)

    # Set the first detected row as the header dynamically
    df.columns = df.iloc[0].tolist()
    df = df[1:].reset_index(drop=True)

    # Drop column indexes that are completely unpopulated empty strings
    df = df.loc[:, (df != "").any(axis=0)]

    return df



# --- Usage Example ---
df = universal_borderless_extractor("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf", page_num=0)
print(df.to_string())


                                                                                                                                                                                                                             GBP Statement
0                                                                                                                                                                                                                Generated on Jul 30, 2026
1                                                                                                                                                                                                                      Revolut Bank UK Ltd
2                                                                                                                                                                                                   PRASAD CHATHURA MANAMALA MUDIYANSELAGE
3                                                           

In [ ]:
# pip install docling
from docling.document_converter import DocumentConverter

# Initialize the converter (Downloads lightweight layout models automatically)
converter = DocumentConverter()

# Convert the document
result = converter.convert("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")

# Docling segments the PDF into semantic pieces (Paragraphs, Tables, Headers)
for element in result.document.tables:
    # 1. Grab the table cleanly structured as a Pandas DataFrame
    df = element.to_dataframe()
    print(df)
    
    # 2. Or view it as perfectly formatted Markdown to verify blank spaces
    # markdown_table = element.to_markdown()


d:\My-Projects\GenAI-Microservices\micro_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-08-01 00:48:11,547 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-01 00:48:11,633 [RapidOCR] download_file.py:60: File exists and is valid: D:\My-Projects\GenAI-Microservices\micro_env\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-01 00:48:11,640 [RapidOCR] main.py:63: Using D:\My-Projects\GenAI-Microservices\micro_env\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-01 00:48:11,755 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-01 00:48:11,786 [RapidOCR] download_file.py:60: File exists and is valid: D:\My-Projects\GenAI-Microservices\micro_env\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[I

In [1]:
import fitz  # PyMuPDF

def extract_spatial_grid(pdf_path, char_width=5.0, line_height=12.0):
    """
    Extracts PDF text while perfectly preserving its visual layout using spaces.
    
    :param char_width: Approximate pixel width of a single character. 
                       Lower = more spaces between words. Higher = words squished together.
    :param line_height: Approximate pixel height of a single line.
    """
    doc = fitz.open(pdf_path)
    full_document_text = []

    for page_num, page in enumerate(doc):
        # 1. Calculate the grid dimensions based on the physical page size
        page_width = page.rect.width
        page_height = page.rect.height
        
        max_cols = int(page_width / char_width) + 1
        max_rows = int(page_height / line_height) + 1
        
        # 2. Initialize a 2D array (matrix) filled entirely with blank spaces
        grid = [[" " for _ in range(max_cols)] for _ in range(max_rows)]
        
        # 3. Extract all words and their coordinates
        words = page.get_text("words")
        
        for w in words:
            x0, y0, x1, y1, text = w[:5]
            
            # Map the physical coordinate to our text grid indices
            row_idx = int(y0 / line_height)
            col_idx = int(x0 / char_width)
            
            # Boundary safety (in case elements render slightly off-page)
            row_idx = min(max(row_idx, 0), max_rows - 1)
            
            # 4. Inject the word's characters into the grid at the exact location
            for i, char in enumerate(text):
                target_col = col_idx + i
                if 0 <= target_col < max_cols:
                    # Overwrite the blank space with the actual character
                    grid[row_idx][target_col] = char
                    
        # 5. Render the 2D grid matrix back into a flat string
        page_strings = []
        for row_list in grid:
            # Join the row into a string and remove trailing whitespace on the right
            line_str = "".join(row_list).rstrip()
            
            # Only append the line if it actually contains text
            if line_str.strip(): 
                page_strings.append(line_str)
                
        full_document_text.append(f"--- PAGE {page_num + 1} ---")
        full_document_text.append("\n".join(page_strings))
        
    return "\n\n".join(full_document_text)

# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_spatial_grid("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")
    print(layout_result)

--- PAGE 1 ---

                                                                                   GBP      Statement
                                                                                           GenerateonJul30,2026
                                                                                                RevoluBankUKLtd
       PRASAD    CHATHURA      MANAMALA       MUDIYANSELAGE
       77 MarlborougGrove                                             SortCode     230120
       Flat33PressCourt                                               AccountNumber66593646
       SE1 5JU
       London                                                         IBAN         GB31REVO23012066593646
                                                                      BIC          REVOGB21
       Balance   summary
        Product                                   Openingbalance   Moneyout        Moneyin               Closing
                                                                 

In [2]:
import fitz  # PyMuPDF

def extract_precise_layout(
    pdf_path: str, 
    y_tolerance: float = 10.0, 
    space_width: float = 5.0
) -> str:
    """
    Extracts PDF text with 100% character accuracy while mimicking the 
    visual layout using dynamically calculated spaces.
    
    :param y_tolerance: Vertical pixels to group words onto the same line.
    :param space_width: The approximate pixel width of a standard space character.
                        Increase this to tighten columns, decrease to spread them.
    """
    doc = fitz.open(pdf_path)
    full_document_text = []

    for page_num, page in enumerate(doc):
        # get_text("words") returns: (x0, y0, x1, y1, "word", block_no, line_no, word_no)
        words = page.get_text("words")
        
        if not words:
            continue
            
        # 1. Group words into horizontal lines
        lines_by_y = {}
        for w in words:
            # Grouping by the top coordinate (y0)
            y_bucket = round(w[1] / y_tolerance) 
            lines_by_y.setdefault(y_bucket, []).append(w)
            
        # 2. Sort the lines top-to-bottom on the page
        sorted_y_buckets = sorted(lines_by_y.keys())
        
        page_strings = []
        
        for y in sorted_y_buckets:
            line_words = lines_by_y[y]
            
            # Sort words left-to-right within the current line
            line_words.sort(key=lambda w: w[0])
            
            line_str = ""
            cursor_x = 0.0
            
            for w in line_words:
                x0, y0, x1, y1, text = w[:5]
                
                # Calculate the physical gap from our cursor to the next word
                gap = x0 - cursor_x
                
                if cursor_x == 0.0:
                    # First word on the line: calculate its left-side indentation
                    num_spaces = max(0, int(x0 / space_width))
                    line_str += " " * num_spaces
                elif gap > 0:
                    # Subsequent words: insert spaces proportional to the physical gap
                    # We use max(1) to ensure at least one space separates distinct words
                    num_spaces = max(1, int(gap / space_width)) 
                    line_str += " " * num_spaces
                else:
                    # Edge case: Words are overlapping or touching directly in the PDF
                    line_str += " "
                    
                # Append the ENTIRE word intact, guaranteeing no lost letters
                line_str += text
                
                # Update cursor to the right-edge of the current word
                cursor_x = x1
                
            # Remove trailing whitespace at the end of the line
            page_strings.append(line_str.rstrip())
            
        full_document_text.append(f"--- PAGE {page_num + 1} ---")
        full_document_text.append("\n".join(page_strings))

    return "\n\n".join(full_document_text)

# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_precise_layout("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")
    print(layout_result)

--- PAGE 1 ---

                                                                                   GBP Statement
                                                                                           Generated on Jul 30, 2026
                                                                                                Revolut Bank UK Ltd
       PRASAD CHATHURA MANAMALA MUDIYANSELAGE
       77 Marlborough Grove                                              Sort Code      230120
       Flat 33, Press Court                                                Account Number 66593646
       SE1 5JU
       London                                                         IBAN         GB31REVO23012066593646
                                                                      BIC          REVOGB21
       Balance summary
                                                                                                         Closing
        Product                                    Opening balanc

In [3]:
# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_precise_layout("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf")
    print(layout_result)

--- PAGE 1 ---

              AMD Developer Hackathon: Participant
              Submission Guide
              This document covers everything you need to build and submit a competitive entry. Exact
              evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
              hardcoded to specific answers.
              Track 1: General-Purpose AI Agent
              What you are building
              An AI agent that handles a wide variety of natural language tasks across multiple capability
              domains, using Fireworks AI models as efficiently as possible.
              Why this task exists:
              Enterprises want to control AI spend without sacrificing user experience: not every task needs a
              premium proprietary model. A common pattern is hosting a range of models in-house
              (open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
              necessary. Track 1 asks y

In [4]:
# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_precise_layout("test_pdfs\\BMC_Startup-Enclave.pdf")
    print(layout_result)

--- PAGE 1 ---

           1. Project Title (Title should express the expected output of the project)
           Enclave: Commercialization of a Secure On-Premise AI Platform for Institutional Knowledge and
           Talent Intelligence
           2. Customer Segment (Please briefly describe for whom the company is giving your product or service and
           Who is the most important customer category of your company?)
                                                                     Maximum 200 words
           Enclave targets organisations that must handle sensitive documents internally and are structurally
           unable to adopt foreign cloud-based AI tools. Three primary segments:
           (i) Public sector. Government ministries, state-owned enterprises, and regulatory bodies that
           manage policy documents, minutes, approvals, circulars, and citizen data under the Personal Data
           Protection Act (No. 9 of 2022) and the ICTA Government Cloud Policy.
   

In [5]:
# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_precise_layout("test_pdfs\\01030000000008.pdf")
    print(layout_result)

--- PAGE 1 ---

           Circulating Things, Circulating Stereotypes                                           73
           ­indicates the use of balsam, which is “indigenous
           in various parts of Arabia,” as an ingredient in the
           “Myrabolan comfit.”25 Such references emphasize
           Arabia’s exoticism and refined taste, as well as the
           sweetness and fragrance of its products, which
           were much valued during a time when the con-
           sumption of sugar and spices was rising rapidly
           among European populations.
             Coffee is another staple thing customarily asso-
           ciated with the area. In his Dictionary, Johnson indi-
           cates the Arabic origin of coffee and rightly so, as
           one the most popular types of coffee is called “Ara-
           bica” because it was first domesticated for commer-
           cial use in the southern part of Arabia the Happy  Figure 4.2 William Hogarth, Taste in High 

In [29]:
import fitz  # PyMuPDF

def extract_precise_layout_with_topics(
    pdf_path: str, 
    y_tolerance: float = 10.0, 
    space_width: float = 5.0,
    column_gap_threshold: float = 40.0,  # Min gap in pixels to be considered a column break
    column_gap_multiplier: int = 1    # How much to multiply the spaces to exaggerate the gap
) -> str:
    """
    Extracts layout precisely using a linear space cursor while converting 
    larger fonts into topics, and exaggerating wide gaps to make columns visible.
    """
    doc = fitz.open(pdf_path)
    full_document_text = []

    for page_num, page in enumerate(doc):
        text_dict = page.get_text("dict")
        lines_by_y = {}
        
        for block in text_dict.get("blocks", []):
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text").strip()
                    if not text:
                        continue
                    
                    bbox = span.get("bbox")
                    font_size = span.get("size")
                    font_name = span.get("font", "").lower()
                    flags = span.get("flags")
                    
                    is_bold = (flags & 2**4) or ("bold" in font_name)
                    
                    y_bucket = round(bbox[1] / y_tolerance)
                    lines_by_y.setdefault(y_bucket, []).append({
                        "x0": bbox[0],
                        "x1": bbox[2],
                        "text": text,
                        "size": font_size,
                        "is_bold": is_bold
                    })
        
        if not lines_by_y:
            continue

        sorted_y_buckets = sorted(lines_by_y.keys())
        page_strings = []
        
        for y in sorted_y_buckets:
            line_spans = lines_by_y[y]
            line_spans.sort(key=lambda s: s["x0"])
            
            max_font_size = max(s["size"] for s in line_spans)
            is_header = max_font_size > 14
            
            line_str = ""
            cursor_x = 0.0
            
            for s in line_spans:
                gap = s["x0"] - cursor_x
                
                if cursor_x == 0.0:
                    num_spaces = max(0, int(s["x0"] / space_width))
                    line_str += " " * num_spaces
                elif gap > 0:
                    num_spaces = max(1, int(gap / space_width))
                    
                    # --- NEW LOGIC: Exaggerate the gap if it exceeds the threshold ---
                    if gap > column_gap_threshold:
                        num_spaces *= column_gap_multiplier
                        
                    line_str += " " * num_spaces
                else:
                    line_str += " "
                
                if s["is_bold"] and not is_header:
                    line_str += f"**{s['text']}**"
                else:
                    line_str += s["text"]
                    
                cursor_x = s["x1"]
            
            if is_header:
                clean_line = line_str.strip()
                if max_font_size > 20:
                    line_str = f"# {clean_line}"
                elif max_font_size > 16:
                    line_str = f"## {clean_line}"
                else:
                    line_str = f"### {clean_line}"
            
            page_strings.append(line_str.rstrip())
            
        full_document_text.append(f"--- PAGE {page_num + 1} ---")
        full_document_text.append("\n".join(page_strings))

    return "\n\n".join(full_document_text)


# --- Execution Example ---
if __name__ == "__main__":
    layout_result = extract_precise_layout_with_topics("test_pdfs\\Participant Guide_ AMD Developer Hackathon (ACT II).pdf")
    print(layout_result)

--- PAGE 1 ---

## AMD Developer Hackathon: Participant
## Submission Guide
              This document covers everything you need to build and submit a competitive entry. Exact
              evaluation inputs are intentionally omitted: your agent must be genuinely capable, not
              hardcoded to specific answers.
### Track 1: General-Purpose AI Agent
              What you are building
              An AI agent that handles a wide variety of natural language tasks across multiple capability
              domains, using Fireworks AI models as efficiently as possible.
              Why this task exists:
              Enterprises want to control AI spend without sacrificing user experience: not every task needs a
              premium proprietary model. A common pattern is hosting a range of models in-house
              (open-source, fine-tuned, RAG-based) and only calling out to a premium API when genuinely
              necessary. Track 1 asks you to build that smart router: r

In [30]:
if __name__ == "__main__":
    layout_result = extract_precise_layout_with_topics("test_pdfs\\01030000000008.pdf")
    print(layout_result)

--- PAGE 1 ---

           Circulating Things, Circulating Stereotypes                                           73
           ­indicates the use of balsam, which is “indigenous
           in various parts of Arabia,” as an ingredient in the
           “Myrabolan comfit.”25 Such references emphasize
           Arabia’s exoticism and refined taste, as well as the
           sweetness and fragrance of its products, which
           were much valued during a time when the con-
           sumption of sugar and spices was rising rapidly
           among European populations.
             Coffee is another staple thing customarily asso-
           ciated with the area. In his Dictionary, Johnson indi-
           cates the Arabic origin of coffee and rightly so, as
           one the most popular types of coffee is called “Ara-
           bica” because it was first domesticated for commer-
           cial use in the southern part of Arabia the Happy  Figure 4.2 William Hogarth, Taste in High 

In [31]:
if __name__ == "__main__":
    layout_result = extract_precise_layout_with_topics("test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf")
    print(layout_result)

--- PAGE 1 ---

# GBP Statement
                                                                                           Generated on Jul 30, 2026
                                                                                                Revolut Bank UK Ltd
       PRASAD CHATHURA MANAMALA MUDIYANSELAGE
       77 Marlborough Grove                                              Sort Code      230120
       Flat 33, Press Court                                                Account Number 66593646
       SE1 5JU
       London                                                         IBAN         GB31REVO23012066593646
                                                                      BIC          REVOGB21
       Balance summary
                                                                                                         Closing
        Product                                    Opening balance    Money out        Money in
                                                  

In [7]:
import re
from langdetect import detect, LangDetectException
from spellchecker import SpellChecker

class TextQualityValidator:
    def __init__(self, expected_language='en'):
        self.expected_language = expected_language
        self.spell = SpellChecker(language=expected_language)
        
    def detect_artifacts(self, text: str) -> bool:
        """Flags text if it contains encoding errors or too many symbols."""
        # Explicit unicode escape sequence for ''
        if "\ufffd" in text:
            return True
            
        alphanumeric_count = sum(c.isalnum() for c in text)
        total_chars = len(text.replace(" ", "").replace("\n", ""))
        
        if total_chars > 0:
            symbol_ratio = (total_chars - alphanumeric_count) / total_chars
            if symbol_ratio > 0.20:
                return True
                
        return False

    def detect_foreign_language(self, text: str) -> bool:
        """Flags text if the primary language does not match expected language."""
        try:
            detected_lang = detect(text)
            return detected_lang != self.expected_language
        except LangDetectException:
            return True 

    def calculate_oov_ratio(self, text: str, threshold: float = 0.15) -> bool:
        """Flags text if Out-Of-Vocabulary word ratio exceeds threshold."""
        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        words = clean_text.split()
        
        # Only check alpha words (skip numbers)
        alpha_words = [w for w in words if w.isalpha()]
        
        if not alpha_words:
            return False
            
        misspelled = self.spell.unknown(alpha_words)
        oov_ratio = len(misspelled) / len(alpha_words)
        
        return oov_ratio > threshold

    def validate(self, extracted_text: str) -> dict:
        if not extracted_text or not extracted_text.strip():
            return {"is_valid": False, "flags": ["Empty extraction"]}

        is_foreign = self.detect_foreign_language(extracted_text)
        has_artifacts = self.detect_artifacts(extracted_text)
        is_garbled = self.calculate_oov_ratio(extracted_text)
        
        flags = []
        if is_foreign: flags.append("Foreign language detected")
        if has_artifacts: flags.append("High symbol density or encoding artifacts")
        if is_garbled: flags.append("High Out-Of-Vocabulary ratio (OCR garble)")
            
        return {
            "is_valid": len(flags) == 0,
            "flags": flags
        }


# --- Execution ---
if __name__ == "__main__":
    validator = TextQualityValidator(expected_language='en')
    
    good_text = "The quick brown fox jumps over the lazy dog. Total balance is 500."
    print("Good Text:", validator.validate(good_text))
    
    bad_ocr = "Th3 qu!ck br0wn f0x jmps ov3r th lzy d0g. T0t@l balnce 1s 5OO."
    print("Bad OCR:", validator.validate(bad_ocr))
    
    foreign_text = "El zorro marrón rápido salta sobre el perro perezoso."
    print("Foreign Text:", validator.validate(foreign_text))

Good Text: {'is_valid': True, 'flags': []}
Bad OCR: {'is_valid': False, 'flags': ['High Out-Of-Vocabulary ratio (OCR garble)']}
Foreign Text: {'is_valid': False, 'flags': ['Foreign language detected', 'High Out-Of-Vocabulary ratio (OCR garble)']}


In [11]:
import fitz  # PyMuPDF
import json
import re
from typing import List, Dict, Any

# ==========================================
# 1. PyMuPDF Text & Topic Extraction
# ==========================================
def extract_precise_layout_with_topics(
    pdf_path: str, 
    y_tolerance: float = 10.0, 
    space_width: float = 5.0,
    column_gap_threshold: float = 40.0,
    column_gap_multiplier: int = 1
) -> str:
    """
    Extracts layout precisely using a linear space cursor while converting 
    larger fonts into topics, and exaggerating wide gaps to make columns visible.
    """
    doc = fitz.open(pdf_path)
    full_document_text = []

    for page_num, page in enumerate(doc):
        text_dict = page.get_text("dict")
        lines_by_y = {}
        
        for block in text_dict.get("blocks", []):
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text").strip()
                    if not text:
                        continue
                    
                    bbox = span.get("bbox")
                    font_size = span.get("size")
                    font_name = span.get("font", "").lower()
                    flags = span.get("flags")
                    
                    is_bold = (flags & 2**4) or ("bold" in font_name)
                    
                    y_bucket = round(bbox[1] / y_tolerance)
                    lines_by_y.setdefault(y_bucket, []).append({
                        "x0": bbox[0],
                        "x1": bbox[2],
                        "text": text,
                        "size": font_size,
                        "is_bold": is_bold
                    })
        
        if not lines_by_y:
            continue

        sorted_y_buckets = sorted(lines_by_y.keys())
        page_strings = []
        
        for y in sorted_y_buckets:
            line_spans = lines_by_y[y]
            line_spans.sort(key=lambda s: s["x0"])
            
            max_font_size = max(s["size"] for s in line_spans)
            is_header = max_font_size > 14
            
            line_str = ""
            cursor_x = 0.0
            
            for s in line_spans:
                gap = s["x0"] - cursor_x
                
                if cursor_x == 0.0:
                    num_spaces = max(0, int(s["x0"] / space_width))
                    line_str += " " * num_spaces
                elif gap > 0:
                    num_spaces = max(1, int(gap / space_width))
                    if gap > column_gap_threshold:
                        num_spaces *= column_gap_multiplier
                    line_str += " " * num_spaces
                else:
                    line_str += " "
                
                if s["is_bold"] and not is_header:
                    line_str += f"**{s['text']}**"
                else:
                    line_str += s["text"]
                    
                cursor_x = s["x1"]
            
            if is_header:
                clean_line = line_str.strip()
                if max_font_size > 20:
                    line_str = f"# {clean_line}"
                elif max_font_size > 16:
                    line_str = f"## {clean_line}"
                else:
                    line_str = f"### {clean_line}"
            
            page_strings.append(line_str.rstrip())
            
        full_document_text.append(f"--- PAGE {page_num + 1} ---")
        full_document_text.append("\n".join(page_strings))

    return "\n\n".join(full_document_text)

# ==========================================
# 2. Whitespace-to-JSON Parser
# ==========================================
import json
import re
from typing import List, Dict, Any

def parse_spatial_text_to_json(
    spatial_text: str, 
    min_table_lines: int = 2,
    column_space_threshold: int = 3  # Min spaces between headers to be considered separate columns
) -> List[Dict[str, Any]]:
    """
    Parses spatially aligned text by anchoring column boundaries to the header row.
    This forgives jagged text alignment and prevents broken column names.
    """
    lines = spatial_text.splitlines()
    tables = []
    current_block = []

    def process_block(block_lines: List[str]) -> Dict[str, Any]:
        if len(block_lines) < min_table_lines:
            return None

        # Standardize line lengths by padding
        max_len = max(len(line) for line in block_lines)
        padded_lines = [line.ljust(max_len) for line in block_lines]

        # 1. Use the FIRST line as the anchor to define column names and boundaries
        header_line = padded_lines[0]
        
        # Regex: Find text chunks separated by 'column_space_threshold' or more spaces
        regex_pattern = rf'(?!\s).+?(?=\s{{{column_space_threshold},}}|$)'
        matches = list(re.finditer(regex_pattern, header_line))

        if len(matches) < 2:
            return None  # Not a table if there's only 1 column

        columns = []
        for i, match in enumerate(matches):
            header_name = match.group().strip()
            # The column starts where the header starts (with a slight margin to the left)
            start_idx = max(0, match.start() - 2) 
            
            # The column ends where the NEXT header starts, or at the end of the line
            end_idx = matches[i+1].start() - 1 if i + 1 < len(matches) else max_len
            
            columns.append({
                "name": header_name if header_name else f"col_{i+1}",
                "start": start_idx,
                "end": end_idx
            })

        # 2. Slice the remaining rows using these anchored boundaries
        data_rows = []
        for line in padded_lines[1:]:
            row_dict = {}
            is_empty_row = True
            
            for col in columns:
                # Extract text within the strict boundary bucket
                cell_value = line[col["start"]:col["end"]].strip()
                row_dict[col["name"]] = cell_value
                if cell_value:
                    is_empty_row = False
                    
            if not is_empty_row:
                data_rows.append(row_dict)

        return {
            "headers": [col["name"] for col in columns],
            "data": data_rows
        }

    # Block grouping logic
    for line in lines:
        # If line has multiple wide gaps, assume it belongs to a table
        if len(re.findall(r'\s{4,}', line.strip())) >= 1 and not line.startswith("#"):
            current_block.append(line)
        else:
            if current_block:
                table_json = process_block(current_block)
                if table_json:
                    tables.append(table_json)
                current_block = []

    # Catch remaining blocks
    if current_block:
        table_json = process_block(current_block)
        if table_json:
            tables.append(table_json)

    return tables

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == "__main__":
    pdf_file = "test_pdfs\\account-statement_2026-06-01_2026-07-30_en-us_0e8120.pdf"  # Replace with your PDF path

    print("1. Extracting spatial text...")
    text_output = extract_precise_layout_with_topics(
        pdf_path=pdf_file,
        # Adjust gap_multiplier if borderless columns are too close together
        column_gap_multiplier=2 
    )
    
    print("\n--- RAW TEXT OUTPUT ---")
    print(text_output)
    
    print("\n2. Parsing spatial text to JSON tables...")
    json_output = parse_spatial_text_to_json(text_output)
    
    print("\n--- JSON TABLE OUTPUT ---")
    # Print the parsed JSON beautifully
    print(json.dumps(json_output, indent=4))

1. Extracting spatial text...

--- RAW TEXT OUTPUT ---
--- PAGE 1 ---

# GBP Statement
                                                                                           Generated on Jul 30, 2026
                                                                                                Revolut Bank UK Ltd
       PRASAD CHATHURA MANAMALA MUDIYANSELAGE
       77 Marlborough Grove                                                                                            Sort Code      230120
       Flat 33, Press Court                                                                                                Account Number 66593646
       SE1 5JU
       London                                                                                                                  IBAN                  GB31REVO23012066593646
                                                                      BIC                    REVOGB21
       Balance summary
                                  